In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType

catalog_name = 'ecommerce'

In [0]:
df_silver_countries = spark.table(f"{catalog_name}.bronze.brz_countries")

df_silver_countries.printSchema()
df_silver_countries.show(10)

## 1. country_code

In [0]:
# 1.1 transformation: slv_countries -> country_code

df_silver_countries = df_silver_countries \
    .withColumnRenamed("country", "country_code")

df_silver_countries = df_silver_countries.withColumn(
    "country_code", F.upper(F.col("country_code"))
    ) \
    .dropDuplicates(["country_code"])


In [0]:
# 1.2 validation: slv_countries -> country

# length is 2

df_silver_countries.filter(
    (F.length(F.col("country_code")) != 2) |
    F.col("country_code").isNull()) \
    .select("country_code") \
    .show()

## 2. country_name

In [0]:
# 2.1 transformation: slv_countries -> country_name

df_silver_countries = df_silver_countries.withColumn(
    "country_name", 
    F.initcap(F.trim(F.col("country_name")))
)

In [0]:
# 2.2 validation: slv_countries -> country_name

df_silver_countries.select("country_name").filter(
    F.col("country_name").isNull()
).show()

## 3. region

In [0]:
# 3.1 transformation: slv_countries -> region
region_map_values = {
    "Asia-Pacific": "Asia Pacific"
}

df_silver_countries = df_silver_countries.replace(region_map_values, subset = "region") 

df_silver_countries = df_silver_countries.withColumn("region", F.initcap(F.upper(F.col("region"))))

In [0]:
# 3.2 validation: slv_countries -> region
valid_regions = ["Asia Pacific", "Europe", "Americas", "Africa"]

df_silver_countries.filter(~F.col("region").isin(valid_regions)).select("region").show()


## 4. sub-region

In [0]:
# 4.1 transformation: slv_countries -> sub_region

df_silver_countries = df_silver_countries.withColumn(
  "sub_region", F.initcap(F.trim(F.col("sub_region")))
)


In [0]:
# 4.2 validation: slv_countries -> sub_region

valid_sub_regions = ["Oceania", "Western Europe", "South America", "North America", "East Asia", "Northern Europe", "Northern Africa", "Western Africa", "South Asia", "Southeast Asia", "Western Asia", "Southern Europe", "Eastern Africa", "Eastern Europe", "Southern Africa"]

df_silver_countries.select("sub_region").filter(
  F.col("sub_region").isNull() |
  (~F.col("sub_region").isin(valid_sub_regions))
).show()

## 5. currency_code

In [0]:
# 5.1 transformation: slv_countries -> currency_code


df_silver_countries = df_silver_countries.withColumn(
    "currency_code", F.upper(F.col("currency_code"))
)

In [0]:
# 5.2 validation: slv_countries -> currency_code

df_silver_countries.select("currency_code").filter(
    F.col("currency_code").isNull() | (F.length(F.col("currency_code")) != 3 )
).show()

In [0]:
# 6.1 transformation: slv_countries -> currency_name
df_silver_countries.withColumn(
    "currency_name", F.initcap(F.col("currency_name"))
    )


In [0]:
# 6.2 validation: slv_countries -> currency_name

df_silver_countries.select("currency_name").distinct().show()


## quarantine & clean data => df_silver_countries

In [0]:
df_silver_countries_clean = df_silver_countries.filter(
    F.col("country_code").isNotNull() & (F.col("country_code") != "") &
    F.col("country_name").isNotNull() & (F.col("country_name") != "") &
    F.col("region").isNotNull() & (F.col("region") != "") & (~F.col("region").isin(valid_regions)) &
    F.col("sub_region").isNotNull() & (F.col("sub_region") != "") & (~F.col("sub_region").isin(valid_sub_regions)) &
    F.col("currency_code").isNotNull() & (F.col("currency_code") != "") & (F.length(F.col("currency_code")) == 3) &
    F.col("currency_name").isNotNull() & (F.col("currency_name") != "")
    )

df_silver_countries_quarantine = df_silver_countries.filter(
    F.col("country_code").isNull() | (F.col("country_code") == "") |
    F.col("country_name").isNull() | (F.col("country_name") == "") |
    F.col("region").isNull() | (F.col("region") == "") | (~F.col("region").isin(valid_regions)) |
    F.col("sub_region").isNull() | (F.col("sub_region") == "") | (~F.col("sub_region").isin(valid_sub_regions)) |
    F.col("currency_code").isNull() | (F.col("currency_code") == "") | (F.length(F.col("currency_code")) != 3) |
    F.col("currency_name").isNull() | (F.col("currency_name") == "")
) \
    .withColumn("rejection_reason",
        F.when(F.col("country_code").isNull() | (F.col("country_code") == ""), "null/empty country_code") 
        .when(F.col("country_name").isNull() | (F.col("country_name") == ""), "null/empty country_name")
        .when(F.col("region").isNull() | (F.col("region") == "") | (~F.col("region").isin(valid_regions)), "null/empty/invalid region")
        .when(F.col("sub_region").isNull() | (F.col("sub_region") == "") | (~F.col("sub_region").isin(valid_sub_regions)), "null/empty/invalid sub_region")
        .when(F.col("currency_code").isNull() | (F.col("currency_code") == "") | (F.length(F.col("currency_code")) != 3), "null/empty/invalid currency_code")
        .when(F.col("currency_name").isNull() | (F.col("currency_name") == ""), "null/empty currency_name")
        .otherwise("invalid")
        )
    

In [0]:
df_silver_countries_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_countries")

df_silver_countries_quarantine.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_countries_quarantine")